In [15]:
import os
import yaml
from collections import defaultdict
from glob import glob
from pprint import pprint
import polars as pl
from src.utils.config import exhaustive_parse_parameters, fetch_experiment_runs

# Exploring All Observed Parameter Combinations
This section aggregates all unique parameter values from previous experiment runs and constructs a `free_parameters` dictionary based on these observed values. We then use `exhaustive_parse_parameters` to enumerate all possible parameter combinations that have been tried so far. This helps in understanding the explored hyperparameter space and identifying any gaps or redundancies in the search.

In [35]:
# Fetch all runs for the model of interest (adjust model name as needed)
print("Fetching experiment runs...")
runs = fetch_experiment_runs({})  # Empty dict fetches all runs; add filters if needed

# Aggregate unique parameter values (ignore id and name columns)
excluded_columns = {
    'id', 'name', 'config', 'random_seed', 'created_at', 'updated_at', 'store_model',
    'tracker', 'log_freq', 'max_epoch', 'evaluation_cutoffs', 'early_stopping_mode',
    'early_stopping_monitor', 'early_stopping_patience', 'early_stopping', 
    'config_collapse_method'
}
parameter_values = {}
for col in runs.columns:
    if col in excluded_columns:
        continue
        
    values = runs.select(col).unique().to_series().to_list()
    parameter_values[col] = sorted(values)

# Build free_parameters dict
free_parameters = {}
for parameter_name, values in parameter_values.items():
    if len(values) == 1:
        free_parameters[parameter_name] = {'value': list(values)[0]}
    else:
        free_parameters[parameter_name] = {'distribution': 'categorical', 'values': values}

# Show the constructed free_parameters
pprint(free_parameters)

# Enumerate all possible combinations
exhaustive_parse_parameters(free_parameters)

Fetched 256 runs...
Fetched 512 runs...
Fetched 768 runs...
Fetched 1024 runs...
Fetched 1280 runs...
Fetched 1536 runs...
Fetched 1792 runs...
Fetched 2048 runs...
Fetched 2304 runs...
Fetched 2560 runs...
Fetched 2816 runs...
Fetched 3072 runs...
Fetched 3328 runs...
Fetched 3382 runs...
{'batch_size': {'value': 16384},
 'embedding_dimension': {'distribution': 'categorical',
                         'values': [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]},
 'embedding_dropout_rate': {'distribution': 'categorical',
                            'values': [0.0,
                                       0.1,
                                       0.2,
                                       0.3,
                                       0.4,
                                       0.5,
                                       0.6,
                                       0.7,
                                       0.8,
                                       0.9]},
 'l1_regularization': {'distribution': 

{'model': 'matrix_factorization',
 'batch_size': 16384,
 'learning_rate': 0.01,
 'l1_regularization': 0.0,
 'l2_regularization': 1e-10,
 'embedding_dimension': 2,
 'embedding_dropout_rate': 0.1,
 'shuffle': False}

# ElasticNet Hyperparameter Search
config file: `configs/hyperparameter_search/mf:elasticnet.yaml`

This section explores the hyperparameter search space for the ElasticNet regularization in matrix factorization models. The configuration file defines the range of values for each parameter, and we use an exhaustive search to evaluate all possible combinations. This helps in identifying the best set of hyperparameters for optimal model performance.

In [33]:
free_parameters = {
    'embedding_dropout_rate': {
        'value': 0.0
    },
    'embedding_dimension': {
        'distribution': 'categorical',
        'values': [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
    },
    'l1_regularization': {
        'distribution': 'categorical',
        'values': [0.0, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4]
    },
    'l2_regularization': {
        'distribution': 'categorical',
        'values': [0.0, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4]
    },
    'learning_rate': {'value': 0.01},
    'shuffle': {
        'distribution': 'categorical', 
        'values': [True, False]
    },
}

In [34]:
exhaustive_parse_parameters(free_parameters)

Fetched 256 runs...
Fetched 512 runs...
Fetched 768 runs...
Fetched 1024 runs...
Fetched 1280 runs...
Fetched 1536 runs...
Fetched 1792 runs...
Fetched 2048 runs...
Fetched 2304 runs...
Fetched 2560 runs...
Fetched 2567 runs...
Parameter space shape: (10, 7, 7, 2)
Existing configurations in the parameter space: 980 / 980
Parameter space are 100.00% occupied.
Average runs per configuration: 2.21


{'embedding_dropout_rate': 0.0,
 'learning_rate': 0.01,
 'embedding_dimension': 4,
 'l1_regularization': 1e-08,
 'l2_regularization': 1e-07,
 'shuffle': True}

In [3]:
exhaustive_parse_parameters(free_parameters)

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/hafidh_rendyanto/.netrc.


Fetched 256 runs...
Fetched 512 runs...
Fetched 768 runs...
Fetched 1024 runs...
Fetched 1280 runs...
Fetched 1536 runs...
Fetched 1792 runs...
Fetched 2048 runs...
Fetched 2304 runs...
Fetched 2560 runs...
Fetched 2567 runs...
Parameter space shape: (10, 7, 7, 2)
Existing configurations in the parameter space: 980 / 980
Parameter space are 100.00% occupied.
Average runs per configuration: 2.21


{'embedding_dropout_rate': 0.0,
 'learning_rate': 0.01,
 'embedding_dimension': 4,
 'l1_regularization': 1e-08,
 'l2_regularization': 1e-07,
 'shuffle': True}

# Dropout Rate and Embedding Dimension

In this section, we analyze the effects of varying the embedding dropout rate and embedding dimension on model performance. By systematically changing these parameters, we can observe their impact on regularization and model capacity, which are crucial for preventing overfitting and improving generalization. The results from this analysis guide the selection of appropriate values for these hyperparameters.

In [6]:
free_parameters = {
    'embedding_dropout_rate': {
        'distribution': 'categorical',
        'values': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    },
    'embedding_dimension': {
        'distribution': 'categorical',
        'values': [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
    },
    'learning_rate': {
        'value': 0.01
    },
    'shuffle': {
        'distribution': 'categorical', 
        'values': [True, False]
    },
}


In [7]:
exhaustive_parse_parameters(free_parameters)

Fetched 256 runs...
Fetched 512 runs...
Fetched 768 runs...
Fetched 1024 runs...
Fetched 1280 runs...
Fetched 1536 runs...
Fetched 1792 runs...
Fetched 2048 runs...
Fetched 2304 runs...
Fetched 2560 runs...
Fetched 2816 runs...
Fetched 3072 runs...
Fetched 3328 runs...
Fetched 3386 runs...
Parameter space shape: (10, 10, 2)
Existing configurations in the parameter space: 200 / 200
Parameter space are 100.00% occupied.
Average runs per configuration: 16.93


{'learning_rate': 0.01,
 'embedding_dropout_rate': 0.1,
 'embedding_dimension': 512,
 'shuffle': True}

# Shuffling Fix
Earlier, there is a bug in the shuffling implementation where the buffer used for shuffling is not large enough to hold all the data, which results in a non-uniform shuffling. This is now fixed by increasing the buffer size to hold all the data, ensuring a proper shuffle.

As part of the fix, we're also rerunning the experiments with the same hyperparameters to ensure that the results are consistent and not affected by the previous shuffling issue. 

The parameter space bellow represent the first batch of experiments that we run with the shuffling fix. We have already run other parameter combination as well.

In [12]:
free_parameters = {
    'embedding_dropout_rate': {
        'value': 0.0
    },
    'embedding_dimension': {
        'distribution': 'categorical',
        'values': [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
    },
    'l1_regularization': {
        'value': 0.0
    },
    'l2_regularization': {
        'distribution': 'categorical',
        'values': [0.0, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2]
    },
    'learning_rate': {'value': 0.01},
    'shuffle': {
        'distribution': 'categorical', 
        'values': [True, False]
    },
}

In [13]:
exhaustive_parse_parameters(free_parameters)

Fetched 256 runs...
Fetched 512 runs...
Fetched 734 runs...
Parameter space shape: (10, 10, 2)
Existing configurations in the parameter space: 200 / 200
Parameter space are 100.00% occupied.
Average runs per configuration: 3.67


{'embedding_dropout_rate': 0.0,
 'l1_regularization': 0.0,
 'learning_rate': 0.01,
 'embedding_dimension': 2,
 'l2_regularization': 1e-09,
 'shuffle': True}